# LAB 08 - TravelOps
## Notebook: 01_validate_gold_health

Purpose:
Validates that the target-specific Gold health table exists and reports a green CI/CD gate.

Business purpose:
Provides a deployable production readiness gate for travel booking operations.

Technical purpose:
Reads Gold health metrics after the Lakeflow pipeline run and fails the job when mandatory health criteria are not met.

Inputs:
- <target_catalog>.<target_schema>.gold_production_health
- <target_catalog>.<target_schema>.gold_daily_booking_revenue
- <target_catalog>.<target_schema>.gold_payment_reconciliation

Outputs:
No data writes. Displays validation evidence and raises on failure.

Tables/files affected:
None. This notebook is validation-only.

Environment variables/widgets used:
`target_catalog` and `target_schema` supplied by the Bundle job.

Creates/modifies data:
No.

Safe to rerun:
Yes. It is read-only.

Dependencies/prerequisites:
The Lakeflow pipeline must have successfully produced the Gold tables.

Expected result:
Exactly one current health row with `health_passed = true`. `health_passed` requires a nonzero booking count, zero duplicate booking IDs, zero negative booking amounts, and zero payment amount mismatches (a completed payment whose total does not equal the booking amount). It does not require every booking to have a completed payment: `no_payment_record_count` and `pending_or_failed_payment_count` are informational only, since a booking with no payment yet or only a pending/failed payment can be a legitimate lifecycle state, not a defect.

Failure behavior:
Any missing table, zero-row table or failed health condition raises an exception and fails the job.

Environment classification:
Runs automatically in personal_dev and personal_prod. Azure PROD execution is intentionally deferred.


### Step 1 - Resolve validation target

This cell reads the target catalog and schema from job widgets. It performs no writes and fails early if the bundle omitted required parameters.

In [ ]:
dbutils.widgets.text("target_catalog", "")
dbutils.widgets.text("target_schema", "")

target_catalog = dbutils.widgets.get("target_catalog")
target_schema = dbutils.widgets.get("target_schema")
if not target_catalog or not target_schema:
    raise ValueError("target_catalog and target_schema widgets are required")


### Step 2 - Resolve the effective Gold schema

This cell checks the configured application target schema. No legacy fallback is used because Personal DEV and PROD now publish to dedicated schemas.

In [ ]:
health_table_name = "gold_production_health"
health_table = f"`{target_catalog}`.`{target_schema}`.`{health_table_name}`"
configured_health_table = f"{target_catalog}.{target_schema}.{health_table_name}"
if not spark.catalog.tableExists(configured_health_table):
    raise AssertionError(f"Could not find configured health table: {configured_health_table}")

print(f"Validating health table: {health_table}")
health_df = spark.table(health_table)
health_rows = health_df.collect()
display(health_df)

if len(health_rows) != 1:
    raise AssertionError(f"Expected exactly one health row, found {len(health_rows)}")


### Step 3 - Enforce health gate

This cell converts the health row into explicit assertions. Failures identify the failed metric so CI/CD evidence is easy to review.

In [ ]:
row = health_rows[0].asDict()
if row["current_booking_count"] <= 0:
    raise AssertionError("Gold current booking count is zero")
if row["duplicate_current_booking_count"] != 0:
    raise AssertionError("Current booking table contains duplicate booking IDs")
if row["invalid_booking_amount_count"] != 0:
    raise AssertionError(f"Current booking table contains negative booking amounts: {row}")
if row["payment_amount_mismatch_count"] != 0:
    raise AssertionError(f"Completed payments do not reconcile to booking amounts: {row}")
if not row["health_passed"]:
    raise AssertionError(f"TravelOps health gate failed: {row}")

print("TravelOps Gold health validation passed")
print(
    "Reconciliation breakdown (informational, not gated): "
    f"no_payment_record_count={row['no_payment_record_count']}, "
    f"pending_or_failed_payment_count={row['pending_or_failed_payment_count']}, "
    f"matched_payment_count={row['matched_payment_count']}"
)
